# Legal Document QA — Step-by-Step Demo

This notebook demonstrates the complete pipeline:
1. Preprocess a legal document
2. Segment it into smart legal chunks (5-stage pipeline)
3. Build a FAISS vector store
4. Query using hybrid BM25 + FAISS retrieval
5. Generate bilingual answers (English + Hindi) using Ollama

**Requirements**: Ollama running locally with `mistral` model pulled.
```bash
ollama pull mistral
```

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
print('Project root:', Path('..').resolve())

## 1. Preprocess the Sample Contract

In [ ]:
from src.segmentation.preprocessor import preprocess

sample_path = Path('../data/sample_contract.txt')
raw_text = sample_path.read_text(encoding='utf-8')

cleaned_text, doc_type = preprocess(raw_text)
print(f'Document type: {doc_type}')
print(f'Cleaned text length: {len(cleaned_text)} chars')
print('\nFirst 500 chars:')
print(cleaned_text[:500])

## 2. Segment the Document (5-Stage Pipeline)

In [ ]:
from src.segmentation.legal_segmenter import segment_document, get_document_structure

chunks = segment_document(cleaned_text, document_name='Sample Service Agreement')
print(f'Total chunks: {len(chunks)}')
print(f'\nSection types found: {set(c.section_type for c in chunks)}')

In [ ]:
# Show a sample chunk
chunk = chunks[3]
print('=== Sample Chunk ===')
print(f'Breadcrumb : {chunk.breadcrumb}')
print(f'Section type: {chunk.section_type}')
print(f'Operative   : {chunk.is_operative}')
print(f'Cross refs  : {chunk.cross_references}')
print()
print(chunk.text)

In [ ]:
# Show document structure
import json
structure = get_document_structure(cleaned_text, 'Sample Service Agreement')

def print_structure(node, indent=0):
    prefix = '  ' * indent
    print(f"{prefix}[{node['level']}] {node['heading']} ({node['section_type']})")
    for child in node['children']:
        print_structure(child, indent + 1)

print_structure(structure)

## 3. Build the Vector Store

> **Note**: This step downloads the BAAI/bge-large-en-v1.5 model on first run (~1.3 GB).

In [ ]:
from src.pipeline.ingest import ingest_document

docs = ingest_document(
    '../data/sample_contract.txt',
    document_name='Sample Service Agreement',
    store_path='/tmp/legal_qa_demo_store',
)
print(f'Ingested {len(docs)} documents into vector store')

## 4. Hybrid Retrieval Test

In [ ]:
from src.vectorstore.store import load_vectorstore
from src.retrieval.retriever import HybridRetriever

vectorstore = load_vectorstore('/tmp/legal_qa_demo_store')
retriever = HybridRetriever(vectorstore=vectorstore, documents=docs, k=3)

query = 'What are the termination conditions?'
retrieved = retriever.retrieve(query)
print(f'Retrieved {len(retrieved)} docs for: "{query}"')
for i, doc in enumerate(retrieved, 1):
    print(f'\n--- Result {i} ---')
    print(f'Section: {doc.metadata.get("breadcrumb", "")}')
    print(doc.page_content[:300])

## 5. Bilingual QA with Ollama

> **Requires**: Ollama running at `http://localhost:11434` with `mistral` pulled.

In [ ]:
from src.pipeline.query import QueryPipeline

pipeline = QueryPipeline(
    store_path='/tmp/legal_qa_demo_store',
    all_docs=docs,
    k=5,
)

questions = [
    'What are the payment terms?',
    'How can this contract be terminated?',
    'What happens to confidential information after the agreement ends?',
    'Who owns the intellectual property created under this agreement?',
]

for q in questions:
    print(f'\n{"="*60}')
    print(f'Q: {q}')
    answer = pipeline.query(q, language='both')
    print(f'\n🇬🇧 English:')
    print(answer.get('english', '(no answer)'))
    print(f'\n🇮🇳 Hindi (हिंदी):')
    print(answer.get('hindi', '(no answer)'))

## 6. English-only Query

In [ ]:
answer = pipeline.query('What is the total contract value?', language='english')
print(answer['english'])
print('\nSources:')
for src in answer['sources']:
    print(f"  - {src['breadcrumb']} [{src['section_type']}]")

## 7. Hindi-only Query

In [ ]:
answer = pipeline.query('इस अनुबंध में भुगतान की शर्तें क्या हैं?', language='hindi')
print(answer['hindi'])